# Phase 1: Baseline Market with Null Traders

Goal: simulate the single-asset market with every trader using the random null rule, then confirm the empirical lag-1 autocorrelation of returns matches the input phi within sampling error.

Reference: section 3.1 of the proposal for the equations, section 3.6 for the intra-period timing.

In [ ]:
# Number of traders.
N = 200

# Number of periods.
T = 5000

# Market-maker price-impact coefficient (equation 5).
mu = 0.05

# Reduced-form residual AR(1) coefficient in the return law (equation 6).
phi = 0.10

# Standard deviation of exogenous news shocks.
sigma_news = 0.01

# Standard deviation of individual null orders (equation 8).
sigma_q = 1.0

# Random seed.
seed = 42

# Output paths.
fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from reflexive_market import simulate
from reflexive_market.metrics import lag1_autocorr

In [ ]:
rng = np.random.default_rng(seed)

In [ ]:
result = simulate.run(
    T=T,
    N=N,
    mu=mu,
    phi=phi,
    sigma_news=sigma_news,
    sigma_q=sigma_q,
    rng=rng,
)

prices = result["prices"]
returns = result["returns"]
demand = result["demand"]

In [ ]:
rho1 = lag1_autocorr(returns)

print(f"input phi:                    {phi:.4f}")
print(f"empirical lag-1 autocorr:     {rho1:.4f}")
print(f"return mean:                  {returns.mean():.6f}")
print(f"return std:                   {returns.std():.6f}")
print(f"aggregate demand std:         {demand.std():.6f}")

## Sample price path

With every trader using the random null rule, the log price is a random walk with a small autoregressive component carried by phi. There is no drift.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(prices)
ax.set_title("Sample log-price path under the null rule")
ax.set_xlabel("Time period")
ax.set_ylabel("Log price")
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_01_price_path.png", dpi=150)
plt.show()

## Sample return series

Returns are roughly stationary with mean near zero. Visible bursts come from the news shock and the random null orders, not from any predictive signal.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(returns, linewidth=0.5)
ax.axhline(0.0, color="k", linewidth=0.5)
ax.set_title("Sample one-period return series")
ax.set_xlabel("Time period")
ax.set_ylabel("Return")
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_01_return_series.png", dpi=150)
plt.show()

## Return distribution

The histogram should look approximately Gaussian, since both the news shock and the per-trader null orders are Gaussian and aggregate demand averages out toward a Gaussian by the central limit theorem.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(returns, bins=60, density=True, alpha=0.7, label="empirical")

xs = np.linspace(returns.min(), returns.max(), 200)
mu_emp = returns.mean()
sd_emp = returns.std()
ys = np.exp(-0.5 * ((xs - mu_emp) / sd_emp) ** 2) / (sd_emp * np.sqrt(2 * np.pi))
ax.plot(xs, ys, "k-", linewidth=1.0, label="Gaussian reference")

ax.set_title("Return distribution with Gaussian reference")
ax.set_xlabel("Return")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_01_return_histogram.png", dpi=150)
plt.show()

## Save numeric summary

Stored as a small npz so the headline numbers can be read back without rerunning the whole notebook.

In [ ]:
np.savez(
    f"{data_dir}/phase_01_baseline.npz",
    prices=prices,
    returns=returns,
    demand=demand,
    phi_input=np.array(phi),
    phi_empirical=np.array(rho1),
    seed=np.array(seed),
)

## Done when

- The price path looks like a random walk with no drift.
- The return series is roughly stationary with mean near zero.
- The return histogram looks roughly Gaussian.
- The empirical lag-1 autocorrelation is close to phi (try a few seeds and a longer T to tighten the estimate).